# USA Regional Sales Analysis (2014 – Feb 2018)
**Exploratory Data Analysis | Python · Pandas · Matplotlib · Seaborn · Plotly**

| | |
|---|---|
| **Original analysis** | samad ([original repository](https://github.com/samad/usa-regional-sales-eda)) |
| **Adapted and maintained by** | Samad ([GitHub](https://github.com/samadbehlim316-design)) |
| **Note** | Shared and adapted with the original author's permission |

---
## Project Summary
This notebook analyses regional sales data from January 2014 to February 2018 (64,104 order lines) through:

- **Data profiling & cleaning:** schema checks, header fixes, type checks, merge validation
- **Univariate & bivariate analysis:** revenue, margin and unit price across products, channels and regions
- **Trend & seasonality:** monthly and calendar-month sales patterns
- **Outlier detection:** extreme unit prices and order values
- **Customer segmentation:** revenue vs. profit margin by customer
- **Geospatial analysis:** state-level revenue choropleth
- **Correlation analysis:** relationships between key numeric metrics

## Problem Statement
Identify the key revenue and profit drivers across products, channels and regions; uncover seasonal trends and outliers; and use the findings to optimise pricing, promotions and market expansion while reducing revenue concentration risk.

## Objectives
1. Identify the top-performing products, channels and regions
2. Uncover seasonal trends and anomalies for better planning
3. Spot pricing and margin risks from outlier transactions
4. Inform pricing, promotion and market-expansion strategy
5. Prepare aggregated views to feed a Power BI dashboard

## 1. Setup & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from matplotlib.ticker import FuncFormatter

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

millions = FuncFormatter(lambda x, pos: f'{x/1e6:.1f}M')

## 2. Data Ingestion

In [ ]:
file_path = '../data/Regional_Sales_Summary.xlsx'

sheets = pd.read_excel(file_path, sheet_name=None)
print(list(sheets.keys()))   # confirm sheet names before indexing

df_sales     = sheets['Sales Orders']
df_customers = sheets['Customers']
df_products  = sheets['Products']
df_regions   = sheets['Regions']
df_state_reg = sheets['State Regions']
df_budgets   = sheets['2017 Budgets']   # budget sheet is for 2017 (not 2024)

## 3. Data Profiling

In [ ]:
for name, frame in {'Sales Orders': df_sales, 'Customers': df_customers, 'Products': df_products,
                    'Regions': df_regions, 'State Regions': df_state_reg, '2017 Budgets': df_budgets}.items():
    print(f'{name:<14} shape: {frame.shape}')
df_sales.head()

**Fix:** the `State Regions` sheet stores its real column names in the first row, so promote that row to the header.

In [ ]:
df_state_reg.columns = df_state_reg.iloc[0]
df_state_reg = df_state_reg[1:].reset_index(drop=True)
df_state_reg.columns.name = None
df_state_reg.head()

In [ ]:
print('Missing values in sales:', df_sales.isnull().sum().sum())
print('Duplicate rows in sales:', df_sales.duplicated().sum())

## 4. Data Wrangling & Merging
Lookup tables are merged with `validate='m:1'` so pandas raises an error if a lookup key is duplicated (which would silently multiply rows). The row count is asserted afterwards.

In [ ]:
df = (
    df_sales
    .merge(df_customers, how='left', left_on='Customer Name Index', right_on='Customer Index', validate='m:1')
    .merge(df_products,  how='left', left_on='Product Description Index', right_on='Index', validate='m:1')
    .merge(df_regions,   how='left', left_on='Delivery Region Index', right_on='id', validate='m:1')
    .merge(df_state_reg[['State Code', 'Region']], how='left', left_on='state_code', right_on='State Code', validate='m:1')
    .merge(df_budgets,   how='left', on='Product Name', validate='m:1')
)

assert len(df) == len(df_sales), 'Merge changed the row count'
df = df.drop(columns=['Customer Index', 'Index', 'id', 'State Code'], errors='ignore')
df.columns = df.columns.str.lower()
df.shape

### 4.1 Select & rename columns

In [ ]:
cols_to_keep = [
    'ordernumber', 'orderdate', 'customer names', 'channel', 'product name',
    'order quantity', 'unit price', 'line total', 'total unit cost',
    'state_code', 'state', 'region', 'latitude', 'longitude', '2017 budgets'
]

# rename() maps all keys at once, so state_code -> state and state -> state_name do not clash
df = df[cols_to_keep].rename(columns={
    'ordernumber': 'order_number', 'orderdate': 'order_date', 'customer names': 'customer_name',
    'product name': 'product_name', 'order quantity': 'quantity', 'unit price': 'unit_price',
    'line total': 'revenue', 'total unit cost': 'cost',
    'state_code': 'state', 'state': 'state_name', 'region': 'us_region',
    'latitude': 'lat', 'longitude': 'lon', '2017 budgets': 'budget'
})

df['order_date'] = pd.to_datetime(df['order_date'])
print(df['order_date'].min(), '->', df['order_date'].max())

# Budgets only apply to 2017 orders
df.loc[df['order_date'].dt.year != 2017, 'budget'] = np.nan
df.info()

## 5. Feature Engineering

In [ ]:
df['total_cost']        = df['quantity'] * df['cost']
df['profit']            = df['revenue'] - df['total_cost']
df['profit_margin_pct'] = df['profit'] / df['revenue'] * 100
df['order_month']       = df['order_date'].dt.to_period('M')
df['order_month_name']  = df['order_date'].dt.month_name()
df['order_month_num']   = df['order_date'].dt.month
df.head()

## 1. Monthly Sales Trend
**Goal:** Track revenue over time to detect seasonality and anomalies.

In [ ]:
monthly = df.groupby('order_month')['revenue'].sum()

ax = monthly.plot(marker='o', color='navy', figsize=(15, 4))
ax.yaxis.set_major_formatter(millions)
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Total Revenue (Millions USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insights**


- Monthly revenue cycles between roughly 21M and 26M USD.
- The overall level is stable year over year, indicating a reliable demand pattern.
- A sharp drop in early 2017 (about 21.2M USD) stands out as an outlier and should be investigated (market disruption, promotion timing or data gaps).

## 2. Seasonality: Revenue by Calendar Month (Excluding 2018)
**Goal:** Show the seasonal pattern by pooling all years per calendar month. 2018 is excluded because only January and February are available, which would skew the totals.

In [ ]:
df_full = df[df['order_date'].dt.year != 2018]

seasonal = (df_full.groupby(['order_month_num', 'order_month_name'])['revenue']
                   .sum().reset_index().sort_values('order_month_num'))

plt.figure(figsize=(13, 4))
plt.plot(seasonal['order_month_name'], seasonal['revenue'], marker='o', color='navy')
plt.gca().yaxis.set_major_formatter(millions)
plt.title('Overall Monthly Sales Trend (Excluding 2018)')
plt.xlabel('Month')
plt.ylabel('Total Revenue (Millions USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insights**


- May is the strongest month (about 102M USD pooled) and February the weakest (about 91.5M USD).
- The spread between peak and trough is modest, so seasonality is mild rather than extreme.
- Plan promotions for the February trough and use the May peak for launches.

## 3. Top 10 Products by Revenue
**Goal:** Identify the highest-grossing products.

In [ ]:
top_prod = (df.groupby('product_name')['revenue'].sum() / 1e6).nlargest(10)

plt.figure(figsize=(9, 4))
sns.barplot(x=top_prod.values, y=top_prod.index, hue=top_prod.index, palette='viridis', legend=False)
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Revenue (Millions USD)')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

**Insights**


- Products 26 and 25 pull ahead (about 110M USD or more each).
- Revenue then drops sharply to the 68–75M USD band, with the bottom of the top 10 at about 52–57M USD.
- The top products carry a large share of sales, which is a concentration risk. Test growth pilots on mid-tier products.

## 4. Top 10 Products by Average Profit per Order Line
**Goal:** Compare average profitability across products.

In [ ]:
top_profit = df.groupby('product_name')['profit'].mean().nlargest(10)

plt.figure(figsize=(9, 4))
sns.barplot(x=top_profit.values, y=top_profit.index, hue=top_profit.index, palette='viridis', legend=False)
plt.title('Top 10 Products by Average Profit per Order Line')
plt.xlabel('Average Profit (USD)')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

**Insights**


- Products 18 and 28 lead, with an average profit of roughly 7.9K–8.0K USD per order line.
- Products 12, 26 and 21 follow in the 7.4K–7.6K USD range.
- The highest-revenue products are not always the most profitable, so use the margin leaders in pricing and mix decisions.

## 5. Sales by Channel
**Goal:** Show how revenue splits across channels.

In [ ]:
chan_sales = df.groupby('channel')['revenue'].sum().sort_values(ascending=False)

plt.figure(figsize=(5, 5))
plt.pie(chan_sales.values, labels=chan_sales.index, autopct='%1.1f%%',
        startangle=140, colors=sns.color_palette('coolwarm', len(chan_sales)))
plt.title('Total Sales by Channel')
plt.tight_layout()
plt.show()

**Insights**


- Wholesale accounts for about 54% of sales, Distributor about 31% and Export about 15%.
- The business leans heavily on domestic bulk channels. Expanding Export (overseas marketing, partners) would diversify revenue.

## 6. Order Value Distribution
**Goal:** Understand typical order values and outliers.

In [ ]:
order_value = df.groupby('order_number')['revenue'].sum()

plt.figure(figsize=(12, 4))
plt.hist(order_value, bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Order Value')
plt.xlabel('Order Value (USD)')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

**Insights**


- The distribution is heavily right-skewed: most orders sit between roughly 0 and 120K USD, with a mode near 60K USD.
- A long tail reaches about 500K USD, but these large orders are a small share of volume. They suit dedicated account management.

## 7. Profit Margin % vs. Unit Price
**Goal:** Check whether margin depends on price.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df['unit_price'], df['profit_margin_pct'], alpha=0.6, color='green')
plt.title('Profit Margin % vs. Unit Price')
plt.xlabel('Unit Price (USD)')
plt.ylabel('Profit Margin (%)')
plt.tight_layout()
plt.show()

print('Correlation:', round(df['unit_price'].corr(df['profit_margin_pct']), 3))

**Insights**


- Margins concentrate between about 18% and 60%, with no clear relationship to unit price (prices range up to about 6,500 USD).
- Horizontal bands indicate consistent margin tiers across the price range, pointing to standardised pricing.
- Points below 18% at both low and high prices may indicate cost or pricing issues worth a closer look.

## 8. Unit Price Distribution per Product
**Goal:** Compare price variability and outliers by product.

In [ ]:
plt.figure(figsize=(12, 4))
sns.boxplot(data=df, x='product_name', y='unit_price', color='g')
plt.title('Unit Price Distribution per Product')
plt.xlabel('Product')
plt.ylabel('Unit Price (USD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Insights**


- Products 8, 17, 27, 20 and 28 show high-price outliers above their upper whiskers, possibly bulk, premium or special-edition orders.
- Products 20 and 27 show very low-price outliers (near 100 USD), which may be promotions or test items.
- Consider excluding such outliers from average-price and margin calculations, and decide whether recurring ones should be formalised as promotions or phased out.

## 9. Total Sales by US Region
**Goal:** Compare regional performance.

In [ ]:
region_sales = df.groupby('us_region')['revenue'].sum().sort_values(ascending=False) / 1e6

plt.figure(figsize=(10, 4))
sns.barplot(x=region_sales.values, y=region_sales.index, hue=region_sales.index, palette='Greens_r', legend=False)
plt.title('Total Sales by US Region', fontsize=16, pad=12)
plt.xlabel('Total Sales (Millions USD)')
plt.ylabel('US Region')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

print((region_sales / region_sales.sum() * 100).round(1))

**Insights**


- The West leads (about 360M USD, roughly 30% of sales), with the South and Midwest each contributing over 320M USD (roughly 26–27% each).
- The Northeast trails at about 210M USD (roughly 17%), a clear growth opportunity for local promotions and partnerships.

## 10. Total Sales by State (Choropleth Map)
**Goal:** Show the geographic distribution of revenue.

In [ ]:
state_sales = df.groupby('state', as_index=False)['revenue'].sum()
state_sales['revenue_m'] = state_sales['revenue'] / 1e6

fig = px.choropleth(
    state_sales, locations='state', locationmode='USA-states', color='revenue_m',
    scope='usa', labels={'revenue_m': 'Total Sales (M USD)'},
    color_continuous_scale='Blues', hover_data={'revenue_m': ':.2f'}
)
fig.update_layout(title_text='Total Sales by State', margin=dict(l=0, r=0, t=40, b=0))
fig.show()

**Insights**


- Revenue is concentrated in a few large states, led by California, with a gradual decline through mid-tier and lower-tier states.
- Under-penetrated states are candidates for targeted growth initiatives.

## 11. Top & Bottom 10 Customers by Revenue
**Goal:** Identify the highest- and lowest-revenue customers.

In [ ]:
cust_rev = df.groupby('customer_name')['revenue'].sum()
top_rev, bottom_rev = cust_rev.nlargest(10), cust_rev.nsmallest(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=top_rev.values / 1e6, y=top_rev.index, hue=top_rev.index, palette='Blues_r', legend=False, ax=axes[0])
axes[0].set_title('Top 10 Customers by Revenue')
sns.barplot(x=bottom_rev.values / 1e6, y=bottom_rev.index, hue=bottom_rev.index, palette='Reds', legend=False, ax=axes[1])
axes[1].set_title('Bottom 10 Customers by Revenue')
for a in axes:
    a.set_xlabel('Revenue (Millions USD)')
    a.set_ylabel('Customer')
plt.tight_layout()
plt.show()

**Insights**


- Aibox Company leads the list, and even the 10th-ranked customer contributes roughly 10–12M USD.
- The bottom group tops out at about 4M USD, roughly half the revenue of the top customer, showing high concentration among top accounts.
- Prioritise retention and upselling for the top ten and run growth campaigns for the lower cohort.

## 12. Average Profit Margin by Channel
**Goal:** Compare profitability across channels.

In [ ]:
channel_margin = df.groupby('channel')['profit_margin_pct'].mean().sort_values(ascending=False)

plt.figure(figsize=(6, 4))
ax = sns.barplot(x=channel_margin.index, y=channel_margin.values, hue=channel_margin.index, palette='coolwarm', legend=False)
for i, v in enumerate(channel_margin.values):
    ax.text(i, v + 0.5, f'{v:.2f}%', ha='center', fontweight='bold')
plt.title('Average Profit Margin by Channel')
plt.xlabel('Sales Channel')
plt.ylabel('Avg Profit Margin (%)')
plt.tight_layout()
plt.show()

**Insights**


- Export leads at 37.93%, followed by Distributor (37.56%) and Wholesale (37.09%).
- The spread is under 1 percentage point, so profitability is consistent across channels. Growing Export volume is the most promising lever.

## 13. Top 10 States by Revenue and Order Count
**Goal:** Compare the highest-revenue states with their order volumes.

In [ ]:
state_rev = (df.groupby('state_name')
               .agg(revenue=('revenue', 'sum'), orders=('order_number', 'nunique'))
               .sort_values('revenue', ascending=False).head(10))

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
sns.barplot(x=state_rev.index, y=state_rev['revenue'] / 1e6, hue=state_rev.index, palette='coolwarm', legend=False, ax=axes[0])
axes[0].set_title('Top 10 States by Revenue')
axes[0].set_ylabel('Revenue (Millions USD)')
sns.barplot(x=state_rev.index, y=state_rev['orders'], hue=state_rev.index, palette='coolwarm', legend=False, ax=axes[1])
axes[1].set_title('Order Count for the Same States')
axes[1].set_ylabel('Orders')
for a in axes:
    a.set_xlabel('State')
plt.tight_layout()
plt.show()

**Insights**


- California leads by a wide margin (about 230M USD and roughly 7,500 orders), more than twice the next state.
- Illinois, Florida and Texas form a second tier with roughly 85–110M USD and 3,600–4,600 orders each.
- Revenue tracks order count closely, implying similar average order values across states. Grow order volume in mid-tier states and order size in weaker ones.

## 14. Customer Segmentation: Revenue vs. Profit Margin
**Goal:** Segment customers by revenue and margin, sized by order volume.

In [ ]:
cust = df.groupby('customer_name').agg(
    total_revenue=('revenue', 'sum'),
    avg_margin=('profit_margin_pct', 'mean'),
    orders=('order_number', 'nunique')
)
cust['total_revenue_m'] = cust['total_revenue'] / 1e6

plt.figure(figsize=(7, 5))
sns.scatterplot(data=cust, x='total_revenue_m', y='avg_margin', size='orders', sizes=(20, 200), alpha=0.7)
plt.title('Customer Segmentation: Revenue vs. Profit Margin')
plt.xlabel('Total Revenue (Millions USD)')
plt.ylabel('Avg Profit Margin (%)')
plt.tight_layout()
plt.show()

**Insights**


- Customers above 10M USD in revenue keep margins of about 36–40%, so scale does not erode profitability.
- Most customers sit between 6M and 10M USD with stable margins (about 34–40%).
- Customers under 6M USD show the widest margin variation (about 33–43%), suggesting more volatile costs or discounting among smaller accounts.

## 15. Correlation Heatmap
**Goal:** Identify relationships between key numeric variables.

In [ ]:
num_cols = ['quantity', 'unit_price', 'revenue', 'cost', 'profit']

plt.figure(figsize=(6, 4))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='viridis')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

**Insights**


- Profit and revenue are strongly correlated (0.87).
- Unit price correlates 0.91 with revenue, 0.79 with profit and 0.94 with cost, so pricing decisions drive both income and expense.
- Quantity has almost no correlation with unit price or cost and only modest correlation with revenue (0.34) and profit (0.30), making volume a secondary driver compared to price.

## 6. Export the Processed Dataset

In [ ]:
os.makedirs('../outputs', exist_ok=True)
df.to_csv('../outputs/Sales_data_EDA_exported.csv', index=False)

## 7. Key Insights
- **Revenue cycle:** monthly revenue stays stable at roughly 21–26M USD over 2014–2018, with a sharp one-off dip in early 2017 that warrants investigation.
- **Seasonality:** mild; May is the strongest month and February the weakest.
- **Channel mix:** Wholesale about 54%, Distributor about 31%, Export about 15%, with near-identical margins (about 37–38%).
- **Products:** Products 26 and 25 lead revenue; Products 18 and 28 lead average profit per order line.
- **Margins:** range from about 18% to 60% with no clear link to unit price.
- **Geography:** California dominates (about 230M USD, roughly 7,500 orders); the Northeast is the weakest region.
- **Customers:** strong concentration among the top accounts.

## 8. Recommendations
1. **Outlier strategy:** exclude or formalise bulk-order and promotional outliers when computing averages.
2. **Margin uplift:** apply the pricing levers of top-profit products to mid and low tiers and review costs on underperformers.
3. **Export growth:** invest in overseas marketing and distribution partnerships.
4. **Seasonal planning:** promote in the February trough, launch in May and investigate the early-2017 dip.
5. **Regional expansion:** target the Northeast and under-penetrated states.
6. **Dashboard prep:** build aggregated tables (time series, channel mix, product performance) for Power BI.

## 9. Credits
- **Original analysis:** samad
- **Adapted and maintained by:** Samad ([github.com/samadbehlim316-design](https://github.com/samadbehlim316-design))

## 10. Project Structure
```
├── data/
│   └── Regional_Sales_Summary.xlsx
├── notebooks/
│   └── EDA_Regional_Sales_Analysis.ipynb
├── outputs/
│   └── Sales_data_EDA_exported.csv
└── README.md
```

## 11. How to Run
```bash
git clone https://github.com/samadbehlim316-design/usa-regional-sales-eda.git
cd usa-regional-sales-eda
pip install pandas numpy matplotlib seaborn plotly openpyxl jupyter
jupyter notebook notebooks/EDA_Regional_Sales_Analysis.ipynb
```
Place `Regional_Sales_Summary.xlsx` in the `data/` folder before running.